## 1. 라이브러리 및 데이터 로드

In [4]:
import pandas as pd
import json
import os
from tqdm import tqdm

source = 'Video_Games'
# Clothing_Shoes_and_Jewelry / Electronics / Video_Games
file_path = f'../data/amazon/{source}/{source}_5.json'

# 저장할 디렉토리 설정
temporal_dir = f'../data/amazon/{source}/temporal_split'
os.makedirs(temporal_dir, exist_ok=True)

print(f"데이터 로드 시작 (메모리 최적화 방식): {file_path}")

# 메모리 폭발을 막기 위해 필요한 4개 컬럼만 각각의 가벼운 리스트로 담기
user_ids = []
item_ids = []
ratings = []
timestamps = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Reading JSON lines"):
        d = json.loads(line)
        user_ids.append(d['reviewerID'])
        item_ids.append(d['asin'])
        ratings.append(d.get('overall', 0))
        timestamps.append(d.get('unixReviewTime', 0))

# 리스트들을 합쳐서 데이터프레임으로 변환
df = pd.DataFrame({
    'user_id': user_ids,
    'item_id': item_ids,
    'rating': ratings,
    'timestamp': timestamps
})

# 메모리 즉시 반환 (매우 중요: OOM 방지)
del user_ids, item_ids, ratings, timestamps

print(f"✅ 원본 데이터 형태: {df.shape}")

데이터 로드 시작 (메모리 최적화 방식): ../data/amazon/Video_Games/Video_Games_5.json


Reading JSON lines: 497577it [00:02, 168310.14it/s]


✅ 원본 데이터 형태: (497577, 4)


## 2. 시간순 정렬 및 Train / Val / Test 분할

In [5]:
# 1. 유저 ID와 시간순으로 오름차순 정렬 (과거 -> 현재)
df = df.sort_values(by=['user_id', 'timestamp'])

# 2. 유저별로 최신 리뷰부터 순위를 매김 (최신=1, 그 직전=2, ...)
df['rank_latest'] = df.groupby('user_id').cumcount(ascending=False) + 1

# 3. 분할 기준 적용 (Leave-One-Out)
train_df = df[df['rank_latest'] > 2].copy()
val_df = df[df['rank_latest'] == 2].copy()
test_df = df[df['rank_latest'] == 1].copy()

# 분할에 사용했던 보조 컬럼 제거
train_df = train_df.drop(columns=['rank_latest'])
val_df = val_df.drop(columns=['rank_latest'])
test_df = test_df.drop(columns=['rank_latest'])

print("✅ 시간순 기반 Leave-One-Out 분할 완료!")

✅ 시간순 기반 Leave-One-Out 분할 완료!


## 3. 분할 결과 검증 및 데이터 저장

In [6]:
# 1. 갯수 확인
total_interactions = len(df)
train_len = len(train_df)
val_len = len(val_df)
test_len = len(test_df)

print(f"[분할 결과 요약]")
print(f"전체 리뷰 수: {total_interactions:,}")
print(f"Train 수: {train_len:,} ({train_len/total_interactions*100:.2f}%)")
print(f"Valid 수: {val_len:,} ({val_len/total_interactions*100:.2f}%)")
print(f"Test 수 : {test_len:,} ({test_len/total_interactions*100:.2f}%)\n")

# 2. 유저 수 검증 (모든 세트에 동일한 유저가 존재하는지 확인)
num_users = df['user_id'].nunique()
assert train_df['user_id'].nunique() == num_users, "Train셋에 누락된 유저가 있습니다."
assert val_df['user_id'].nunique() == num_users, "Val셋에 누락된 유저가 있습니다."
assert test_df['user_id'].nunique() == num_users, "Test셋에 누락된 유저가 있습니다."
print("✅ 유저 수 무결성 검증 통과!\n")

# 3. 데이터 저장 (.csv)
# 변수명 에러 방지를 위해 temporal_dir 로 통일하여 저장
train_df.to_csv(os.path.join(temporal_dir, 'train.csv'), index=False)
val_df.to_csv(os.path.join(temporal_dir, 'val.csv'), index=False)
test_df.to_csv(os.path.join(temporal_dir, 'test.csv'), index=False)

print(f"💾 데이터 저장 완료 위치: {temporal_dir}/")

[분할 결과 요약]
전체 리뷰 수: 497,577
Train 수: 387,131 (77.80%)
Valid 수: 55,223 (11.10%)
Test 수 : 55,223 (11.10%)

✅ 유저 수 무결성 검증 통과!

💾 데이터 저장 완료 위치: ../data/amazon/Video_Games/temporal_split/
